In [1]:
from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [2]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

FIELD VALIDATOR

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
sys_msg = SystemMessage("tell me the name of the person and their job role")
d1 = """Tanishpreet is a Data scientist at IDS. Her email is tanishpreet.k@idsil.com"""
d2 = """Tanishpreet is a Data scientist at IDS. Her email is tanishpreet.k@gmail.com"""
hum_msg1 = HumanMessage(d1)
hum_msg2 = HumanMessage(d2)

In [4]:
from pydantic import BaseModel, field_validator

class PersonInfoModel(BaseModel):
    name: str
    work: str
    email: str
      
    @field_validator("email")
    @classmethod
    def check_email(cls, v):
        if v.endswith("idsil.com"):
            return v
        else:
            raise ValueError("Email domain must be idsil.com")

In [5]:
structured_model = model.with_structured_output(PersonInfoModel)

In [6]:
ans1 =  structured_model.invoke([sys_msg, hum_msg1])
ans1

PersonInfoModel(name='Tanishpreet', work='Data scientist', email='tanishpreet.k@idsil.com')

In [7]:
ans2 =  structured_model.invoke([sys_msg, hum_msg2])
ans2

OutputParserException: Failed to parse PersonInfoModel from completion {"name": "Tanishpreet", "work": "Data scientist at IDS", "email": "tanishpreet.k@gmail.com"}. Got: 1 validation error for PersonInfoModel
email
  Value error, Email domain must be idsil.com [type=value_error, input_value='tanishpreet.k@gmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

MODEL VALIDATOR

In [8]:
from typing_extensions import Self
from pydantic import BaseModel, model_validator, ValidationError

class UserModel(BaseModel):
    username: str
    password: str
    password_repeat: str

    @model_validator(mode="after")
    def check_passwords_match(self) -> Self:
        if self.password != self.password_repeat:
            raise ValueError("Passwords do not match")
        return self


try:
    user1 = UserModel(
        username="tanish",
        password="abc123",
        password_repeat="abc123"
    )
    print("User created successfully:")
    print(user1)
except ValidationError as e:
    print("Validation Error:", e)

try:
    user2 = UserModel(
        username="tanish",
        password="abc123",
        password_repeat="xyz123"
    )
    print("User created successfully:")
    print(user2)
except ValidationError as e:
    print("Validation Error:")
    print(e)

User created successfully:
username='tanish' password='abc123' password_repeat='abc123'
Validation Error:
1 validation error for UserModel
  Value error, Passwords do not match [type=value_error, input_value={'username': 'tanish', 'p...sword_repeat': 'xyz123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


AFTER VALIDATOR

In [9]:
class Product(BaseModel):
    price: float

    @field_validator("price", mode="after")
    @classmethod
    def check_positive(cls, value):
        if value <= 0:
            raise ValueError("Price must be positive")
        return value

In [10]:
Product(price=100)

Product(price=100.0)

In [11]:
Product(price=-50)

ValidationError: 1 validation error for Product
price
  Value error, Price must be positive [type=value_error, input_value=-50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

BEFORE VALIDATOR

In [12]:
class User(BaseModel):
    age: int

    @field_validator("age", mode="before")
    @classmethod
    def convert_string_to_int(cls, value):
        print("Before validator received:", value)
        return int(value)
    
user = User(age=21)
print(user)

Before validator received: 21
age=21


PLAIN VALIDATOR

In [14]:
class Model(BaseModel):
    number: int

    @field_validator("number", mode="plain")
    @classmethod
    def custom_validation(cls, value):
        if isinstance(value, str):
            return int(value) * 2
        return value
    
Model(number="19")

Model(number=38)

WRAP VALIDATOR

In [17]:
class Model(BaseModel):
    number: int

    @field_validator("number", mode="wrap")
    @classmethod
    def wrap_validator(cls, value, handler):
        print("Before validation:", value)

        validated_value = handler(value)

        print("After validation:", validated_value)

        return validated_value
    
Model(number="5")

Before validation: 5
After validation: 5


Model(number=5)